In [ ]:
#Se instala las librerías necesarias
%pip install selenium pandas openpyxl webdriver-manager

In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

# 1. Se configura  el servicio para que descargue el driver automáticamente
service = Service(ChromeDriverManager().install())

# 2. Se inicia el navegador (Chrome)
driver = webdriver.Chrome(service=service)

# 3. Se maximiza la ventana para que Selenium "vea" todos los elementos
driver.maximize_window()

In [3]:
# 1. Se importa lo necesario para las esperas
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# 2. Se define la URL de inicio (Asegúrate de que el driver esté abierto)
url_principal = "https://admision.unmsm.edu.pe/Website20262/A/A.html"
driver.get(url_principal)

# 3. Espera explícita: Aguardamos a que la tabla cargue en el DOM
wait = WebDriverWait(driver, 20)
# Se usa el selector que refleja la jerarquía que se encontró:
# table.table-striped busca la tabla, y el espacio + 'a' busca los links dentro
selector_carreras = "table.table-striped tbody tr td a"
enlaces_carreras = wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, selector_carreras)))

# 4. Se guarda la información en una lista de diccionarios
lista_carreras = []
for e in enlaces_carreras:
    lista_carreras.append({
        "carrera": e.text.strip(),
        "url": e.get_attribute("href")  # Esto extrae el link del atributo href
    })

# 5. Se muestra la cantidad de carreras encontradas
print(f"Se han detectado {len(lista_carreras)} carreras.")

# 6. Le echamos un ojito al diccionario que contiene a las carreras con sus respectivo URL
lista_carreras

# Con este código se puede comprobar si la extracción fue correcta
# var_name= [c for c in lista_carreras if "Aquí pon el nombre de la carrera" in c['carrera']]
#print(f"La carrera {var_name}" fue encontrada.)

Se han detectado 111 carreras.


[{'carrera': 'ADMINISTRACIÓN',
  'url': 'https://admision.unmsm.edu.pe/Website20262/A/091/results.html'},
 {'carrera': 'ADMINISTRACIÓN - CHILCA',
  'url': 'https://admision.unmsm.edu.pe/Website20262/A/0914/results.html'},
 {'carrera': 'ADMINISTRACIÓN - HUARAL',
  'url': 'https://admision.unmsm.edu.pe/Website20262/A/0912/results.html'},
 {'carrera': 'ADMINISTRACIÓN - S.J.L',
  'url': 'https://admision.unmsm.edu.pe/Website20262/A/0911/results.html'},
 {'carrera': 'ADMINISTRACIÓN - VILLA RICA',
  'url': 'https://admision.unmsm.edu.pe/Website20262/A/0915/results.html'},
 {'carrera': 'ADMINISTRACIÓN DE LA GASTRONOMÍA',
  'url': 'https://admision.unmsm.edu.pe/Website20262/A/094/results.html'},
 {'carrera': 'ADMINISTRACIÓN DE NEGOCIOS INTERNACIONALES',
  'url': 'https://admision.unmsm.edu.pe/Website20262/A/093/results.html'},
 {'carrera': 'ADMINISTRACIÓN DE NEGOCIOS INTERNACIONALES - HUARAL',
  'url': 'https://admision.unmsm.edu.pe/Website20262/A/0932/results.html'},
 {'carrera': 'ADMINISTRAC

In [4]:
import time
import pandas as pd
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC

def extraer_postulantes_carrera(driver, url_carrera, nombre_carrera):
    """
    Navega por la carrera, cambia a 100 registros y usa el botón 'Siguiente'
    hasta que se deshabilite, capturando datos de atributos data-score/merit.
    """
    driver.get(url_carrera)
    wait = WebDriverWait(driver, 25) # Espera robusta para San Marcos
    
    # 1. Intentar poner la visualización en 100 registros (ID: dt-length-0)
    try:
        select_element = wait.until(EC.element_to_be_clickable((By.ID, "dt-length-0")))
        select = Select(select_element)
        select.select_by_value("100")
        time.sleep(5) # Tiempo para que DataTables procese el cambio
    except:
        print(f"No se pudo cambiar a 100 registros en {nombre_carrera}")

    resultados_carrera = []
    
    while True:
        # 2. Esperar a que la tabla de la HOJA ACTUAL cargue
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "#tablaPostulantes tbody tr")))
        filas = driver.find_elements(By.CSS_SELECTOR, "#tablaPostulantes tbody tr")
        
        # 3. Extraer datos de la hoja en la que estamos
        for fila in filas:
            columnas = fila.find_elements(By.TAG_NAME, "td")
            if len(columnas) >= 6:
                # SOLUCIÓN DATOS VACÍOS: Extraemos de los atributos del HTML
                puntaje = columnas[3].get_attribute("data-score")
                merito = columnas[4].get_attribute("data-merit")
                
                resultados_carrera.append({
                    "Carrera": nombre_carrera,
                    "Código": columnas[0].text.strip(),
                    "Apellidos y Nombres": columnas[1].text.strip(),
                    "Escuela": columnas[2].text.strip(),
                    "Puntaje": puntaje if puntaje else "0.000",
                    "Mérito E.P.": merito if merito else "",
                    "Observación": columnas[5].text.strip()
                })

        # 4. LÓGICA DE MOVIMIENTO: Clic en 'Siguiente' (Paginación)
        try:
            # Identificamos el li que envuelve al botón Siguiente
            # El selector li.dt-paging-button:has(button.next) es el más preciso para tu HTML
            li_siguiente = driver.find_element(By.CSS_SELECTOR, "li.dt-paging-button:has(button.next)")
            
            # Si el contenedor li tiene la clase 'disabled', paramos el bucle
            if "disabled" in li_siguiente.get_attribute("class"):
                print(f"Fin de {nombre_carrera} alcanzado. Registros: {len(resultados_carrera)}")
                break
            
            # Si no está disabled, buscamos el botón interno y clickeamos con JS
            boton_next = li_siguiente.find_element(By.CSS_SELECTOR, "button.next")
            driver.execute_script("arguments[0].click();", boton_next)
            
            # ESPERA CRUCIAL: Pausa para que el contenido de la tabla cambie realmente
            time.sleep(4) 
            
        except Exception as e:
            print(f"Terminando paginación por seguridad en {nombre_carrera}")
            break
            
    return resultados_carrera

In [5]:
import os

# 1. Aseguramos que la lista 'todos_los_datos' empiece limpia
todos_los_datos = []

# 2. BUCLE MAESTRO: Iteramos por TODAS las carreras de 'lista_carreras'
# (Eliminamos el [:2] para que recorra la lista completa)
for i, carrera_info in enumerate(lista_carreras):
    print(f"[{i+1}/{len(lista_carreras)}] Procesando: {carrera_info['carrera']}")
    
    try:
        # Llamamos a tu función corregida
        datos = extraer_postulantes_carrera(driver, carrera_info['url'], carrera_info['carrera'])
        
        # Consolidamos los datos encontrados
        todos_los_datos.extend(datos)
        print(f"Éxito: {len(datos)} registros capturados.")
        
    except Exception as e:
        print(f"Error en {carrera_info['carrera']}: {e}")
        # Si falla una, continuamos con la siguiente para no perder el progreso
        continue

# 3. CONSOLIDACIÓN FINAL EN PANDAS
if todos_los_datos:
    df_final = pd.DataFrame(todos_los_datos)
    
    # 4. EXPORTACIÓN A EXCEL (Carpeta output/)
    output_path = "output/resultados_sanmarcos.xlsx"
    
    # Usamos index=False para que no cree la columna de números a la izquierda
    df_final.to_excel(output_path, index=False)
    
    print("-" * 30)
    print(f"¡PROCESO COMPLETADO!")
    print(f"Total de registros consolidados: {len(df_final)}")
    print(f"Archivo guardado en: {output_path}")
else:
    print("No se recolectó ninguna información. Revisa el driver.")

# 5. Cerramos el navegador al terminar todo
driver.quit()

[1/111] Procesando: ADMINISTRACIÓN
Fin de ADMINISTRACIÓN alcanzado. Registros: 553
Éxito: 553 registros capturados.
[2/111] Procesando: ADMINISTRACIÓN - CHILCA
Fin de ADMINISTRACIÓN - CHILCA alcanzado. Registros: 33
Éxito: 33 registros capturados.
[3/111] Procesando: ADMINISTRACIÓN - HUARAL
Fin de ADMINISTRACIÓN - HUARAL alcanzado. Registros: 31
Éxito: 31 registros capturados.
[4/111] Procesando: ADMINISTRACIÓN - S.J.L
Fin de ADMINISTRACIÓN - S.J.L alcanzado. Registros: 44
Éxito: 44 registros capturados.
[5/111] Procesando: ADMINISTRACIÓN - VILLA RICA
Fin de ADMINISTRACIÓN - VILLA RICA alcanzado. Registros: 33
Éxito: 33 registros capturados.
[6/111] Procesando: ADMINISTRACIÓN DE LA GASTRONOMÍA
Fin de ADMINISTRACIÓN DE LA GASTRONOMÍA alcanzado. Registros: 127
Éxito: 127 registros capturados.
[7/111] Procesando: ADMINISTRACIÓN DE NEGOCIOS INTERNACIONALES
Fin de ADMINISTRACIÓN DE NEGOCIOS INTERNACIONALES alcanzado. Registros: 805
Éxito: 805 registros capturados.
[8/111] Procesando: ADMINI